In [ ]:
import hashlib
import json
from pathlib import Path

import pandas as pd

from utils_firerisk import (
    REGION, SEASON, add_lag_features, build_panel, calendar_features, cluster_events, daily_from_hourly,
    load_oni, nesterov_series, sealed_start, season_label, summarize_events,
)

DATA = Path("../data")
RAW, INTERIM, PROCESSED = DATA / "raw", DATA / "interim", DATA / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

START_DATE, END_DATE = "2012-01-20", "2025-12-31"
PRODUCT = "VIIRS_SNPP_SP"
TREE_FRAC_MIN = 0.5
BIG_PIXELS, BIG_DAYS = 5, 2
A0_GRID = [5, 6, 7, 8, 9, 10]
SEALED_LAST_N = 2
CLIM_END = "2023-12-31"
RUN_SUBSET = None

In [ ]:
PROV = pd.read_csv(INTERIM / "prov_table.csv")
static = pd.read_csv(INTERIM / "static_prov.csv")
if RUN_SUBSET:
    PROV = PROV[PROV["key"].isin(RUN_SUBSET)].reset_index(drop=True)

HOTSPOTS_FILE = INTERIM / "hotspots_enriched.csv"
pts = pd.read_csv(HOTSPOTS_FILE, parse_dates=["ltime", "ldate"])
pts = pts[pts["prov"].isin(PROV["key"])].copy()
pts["forest"] = pts["tree_frac"] >= TREE_FRAC_MIN
pts["conf_ok"] = pts["confidence"].astype(str).str.lower().isin(["n", "h", "nominal", "high"])
print(len(PROV), "tỉnh |", len(pts), "điểm cháy | tỉnh có đặc trưng tĩnh:", int(PROV["key"].isin(static["key"]).sum()))

In [ ]:
events, forest_hotspots = {}, {}
for key in PROV["key"]:
    selected = pts[(pts["prov"] == key) & pts["forest"] & pts["conf_ok"]]
    clustered = cluster_events(selected[["latitude", "longitude", "ldate", "frp"]])
    forest_hotspots[key] = clustered
    events[key] = summarize_events(clustered)
print(pd.Series({k: len(v) for k, v in events.items()}).describe().round(1).to_dict())

Số sự kiện cháy rừng mỗi tỉnh: trung bình 3.689, trung vị 2.173, thấp nhất 1 và cao nhất 16.124. Chênh lệch rất lớn giữa tỉnh có rừng và tỉnh đồng bằng.

In [ ]:
WX_COLUMNS = ["time", "temperature_2m", "dew_point_2m", "precipitation"]


def load_weather(key):
    years = range(pd.Timestamp(START_DATE).year - 1, pd.Timestamp(END_DATE).year + 1)
    frames = [pd.read_csv(RAW / "wx" / f"{key}_{year}.csv", usecols=WX_COLUMNS) for year in years]
    return daily_from_hourly(pd.concat(frames, ignore_index=True))


weather = {key: load_weather(key) for key in PROV["key"]}
days_per_province = len(next(iter(weather.values())))
missing_t13 = int(sum(w["T13"].isna().sum() for w in weather.values()))
print("Thời tiết ngày:", len(weather), "tỉnh | ngày mỗi tỉnh:", days_per_province, "| thiếu T13:", missing_t13)

In [ ]:
def add_nesterov(d):
    d = d.copy()
    for a0 in A0_GRID:
        for mode in ["linear", "binary"]:
            for name, column in [("Dsat", "D13_hpa"), ("Ddew", "dd13_c")]:
                d[f"P_{mode}{a0}_{name}"] = nesterov_series(d["T13"], d[column], d["rain_cal"], a0, mode)
    d["P_alt19_linear7_Dsat"] = nesterov_series(d["T13"], d["D13_hpa"], d["rain_19_19"], 7, "linear")
    d["P_alt19_binary5_Dsat"] = nesterov_series(d["T13"], d["D13_hpa"], d["rain_19_19"], 5, "binary")
    return d


for key in weather:
    weather[key] = add_lag_features(add_nesterov(weather[key]), CLIM_END)

oni = load_oni(INTERIM / "oni.csv")
print("ONI:", oni.index.min().date(), "->", oni.index.max().date())


def oni_columns(index):
    months = pd.DatetimeIndex(index).to_period("M").to_timestamp()
    return oni.reindex(months).values, oni.shift(3).reindex(months).values

Biến thể đúng của chỉ số Nesterov sẽ do bước tái tạo ví dụ mẫu (Cổng 0) chọn, nên bảng giữ cả 26 biến thể.

In [ ]:
STATIC_COLUMNS = ["area_km2", "forest_share", "crop_share", "grass_share", "built_share", "rep_elev_m", "rep_lat", "rep_lon"]
win0, win1 = pd.Timestamp(START_DATE), pd.Timestamp(END_DATE)

dev_parts, sealed_parts, info = [], [], []
for _, prov in PROV.iterrows():
    key, region = prov["key"], prov["region"]
    months = SEASON[region]

    hot_all = pts[(pts["prov"] == key) & pts["conf_ok"]]
    panel = build_panel(weather[key].loc[win0:win1], hot_all, forest_hotspots[key], events[key], BIG_PIXELS, BIG_DAYS)
    panel = pd.concat([panel, calendar_features(panel.index)], axis=1)
    panel["oni"], panel["oni_lag3"] = oni_columns(panel.index)
    panel["season"] = [season_label(t, months) for t in panel.index]
    panel["in_season"] = panel["season"].notna().astype(int)

    province_static = static[static["key"] == key].iloc[0]
    for column in STATIC_COLUMNS:
        panel[column] = province_static[column]
    panel.insert(0, "prov", key)
    panel.insert(1, "region", region)

    sealed_from = sealed_start(months, win0, win1, SEALED_LAST_N)
    panel["sealed"] = (panel.index >= sealed_from) if sealed_from is not None else False
    info.append({"prov": key, "region": region, "sealed_from": sealed_from.date() if sealed_from is not None else None, "rows": len(panel)})
    dev_parts.append(panel[~panel["sealed"]])
    sealed_parts.append(panel[panel["sealed"]])

dev = pd.concat(dev_parts).reset_index()
sealed = pd.concat(sealed_parts).reset_index()
dev.to_csv(PROCESSED / "panel_dev.csv", index=False)
sealed.to_csv(PROCESSED / "panel_SEALED_DO_NOT_OPEN.csv", index=False)

print(pd.DataFrame(info).head(10).to_string(index=False))
print("dev:", dev.shape, "| sealed:", sealed.shape, "(chỉ in kích thước)")
del sealed

dev có 273.209 dòng và sealed 47.776 dòng, 78 cột. Tổng 63 tỉnh x 5.095 ngày = 320.985 dòng. Mốc niêm phong lệch theo vùng (An Giang 2023-12-01, Bắc Giang 2023-11-01, Bình Định 2024-01-01).

In [ ]:
q = pts.copy()
q["year"] = q["ltime"].dt.year
q["region"] = q["prov"].map(REGION)

print("Giờ địa phương trung vị của lần bay qua, theo năm")
print(q.groupby(["year", "daynight"])["lhour"].median().unstack().round(1).T.to_string())

print("\nTỉ lệ độ tin cậy theo năm")
print(q.groupby("year")["confidence"].value_counts(normalize=True).unstack().round(3).to_string())

print("\nLớp phủ tại điểm cháy theo vùng (10 rừng, 40 nông nghiệp, 30 cỏ, 20 cây bụi)")
print(q.groupby("region")["lc_center"].value_counts(normalize=True).unstack().round(3).fillna(0).to_string())

print("\nĐiểm cháy theo tháng và vùng")
print(q[q["conf_ok"]].groupby(["region", q["ltime"].dt.month]).size().unstack(fill_value=0).to_string())

print("\nTỉ lệ thiếu thời tiết trong dev theo vùng")
print(dev.groupby("region")[["T13", "D13_hpa", "rain_cal", "wx_missing"]].apply(lambda g: g.isna().mean()).round(4).to_string())

gate2 = q[q["forest"] & q["conf_ok"]].groupby(["prov", "year"]).agg(
    hotspot_days=("ldate", "nunique"), hotspots=("ldate", "size")
).reset_index()
gate2.to_csv(PROCESSED / "gate2_province_year.csv", index=False)

Giờ vệ tinh bay qua ổn định suốt 14 năm (ban ngày 13h, ban đêm 1h), không có dấu hiệu trôi quỹ đạo. Tỉ lệ độ tin cậy cũng ổn định (n khoảng 86 đến 92%, h 3 đến 4%, l 5 đến 10%), riêng l thấp bất thường ở 2022 và 2025. Rừng chiếm 78 đến 83% điểm cháy ở Bắc Trung Bộ và Đông Bắc, 74% ở Nam Trung Bộ, 50 đến 55% ở Tây Bắc và Tây Nguyên. Đồng bằng sông Cửu Long 87% là đất nông nghiệp, đồng bằng sông Hồng 46% là đô thị. Thời tiết không thiếu giá trị nào ở T13, D13 và mưa trong cả 8 vùng.

In [ ]:
def sha256_prefix(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()[:16]


tracked = {
    "hotspots_enriched.csv": HOTSPOTS_FILE,
    "panel_dev.csv": PROCESSED / "panel_dev.csv",
    "panel_SEALED_DO_NOT_OPEN.csv": PROCESSED / "panel_SEALED_DO_NOT_OPEN.csv",
    "gate2_province_year.csv": PROCESSED / "gate2_province_year.csv",
    "static_prov.csv": INTERIM / "static_prov.csv",
}
settings = {
    "created": pd.Timestamp.now("UTC").isoformat(),
    "product": PRODUCT,
    "start": START_DATE,
    "end": END_DATE,
    "tree_frac_min": TREE_FRAC_MIN,
    "big_pixels": BIG_PIXELS,
    "big_days": BIG_DAYS,
    "a0_grid": json.dumps(A0_GRID),
    "n_provinces": len(PROV),
    "run_subset": json.dumps(RUN_SUBSET),
    "n_hotspots_in_provinces": len(pts),
    "rows_dev": len(dev),
}
manifest = pd.DataFrame(
    [("setting", k, v) for k, v in settings.items()]
    + [("sha256", name, sha256_prefix(path)) for name, path in tracked.items() if path.exists()],
    columns=["type", "name", "value"],
)
manifest.to_csv(PROCESSED / "manifest.csv", index=False)
manifest